# 06 — Model Exploration

Compare **DistanceESPN** and **AttentionESPN** on the three benchmark proteins.
Each protein is processed as its own batch (no collating).

**Reports per model:**
- Per-layer parameter counts
- Peak VRAM per protein (for batch size planning)
- Estimated max batch size for common GPU sizes
- Training time (total + mean per epoch)
- Training loss curves (mean across proteins + per-protein breakdown)

No checkpoints are saved.

In [ ]:
import sys
import time
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent.parent))

try:
    import psutil
    _HAS_PSUTIL = True
except ImportError:
    _HAS_PSUTIL = False
    print("psutil not found — RAM tracking disabled.  pip install psutil")

from src.data.graph_builder import build_graph
from src.data.transform import NormalizeESP
from src.models.distance_espn import DistanceESPN
from src.models.attention_espn import AttentionESPN
from src.training.loss import ESPLoss
from src.utils.config import get_data_root

In [ ]:
DATA_ROOT = get_data_root()
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROTEIN_IDS = [
    "AF-P01082-F1",   # small   ~587 atoms,  ~380 query pts
    "AF-Q16613-F1",   # medium  ~3,270 atoms, ~1,524 query pts
    "AF-B1WC58-F1",   # large   ~14,006 atoms, ~10,748 query pts
]

# Graph construction
VARIANT     = "interp"
SAMPLE_FRAC = 0.05

# Model architecture (shared defaults)
HIDDEN_DIM        = 128
N_RBF             = 16
N_HEADS           = 4      # attention model only
N_COV_SUPP_ROUNDS = 2
N_AQ_ROUNDS       = 3
N_QQ_ROUNDS       = 2

# Training
N_EPOCHS       = 30
LR             = 3e-4
WEIGHT_DECAY   = 1e-4
PEARSON_WEIGHT = 0.1
CLIP_GRAD      = 1.0

# GPU sizes to estimate batch capacity for
GPU_SIZES_GB = [8, 16, 24, 40, 80]

print(f"Device    : {DEVICE}")
if torch.cuda.is_available():
    total_vram = torch.cuda.get_device_properties(DEVICE).total_memory / 1024**3
    print(f"GPU VRAM  : {total_vram:.1f} GB  ({torch.cuda.get_device_name(DEVICE)})")
print(f"Proteins  : {PROTEIN_IDS}")
print(f"Epochs    : {N_EPOCHS}  |  LR: {LR}  |  hidden_dim: {HIDDEN_DIM}")

## 1. Graph Construction

Build one graph per protein, normalize ESP targets across all three, then move to the compute device.

In [ ]:
graphs = []
t_total = time.perf_counter()

for pid in PROTEIN_IDS:
    print(f"\nBuilding graph: {pid}")
    t0 = time.perf_counter()
    g  = build_graph(
        pid, DATA_ROOT,
        variant     = VARIANT,
        sample_frac = SAMPLE_FRAC,
    )
    elapsed = time.perf_counter() - t0
    cov_e  = g["atom", "cov",  "atom" ].edge_index.shape[1]
    supp_e = g["atom", "supp", "atom" ].edge_index.shape[1]
    aq_e   = g["atom", "aq",   "query"].edge_index.shape[1]
    qq_e   = g["query", "qq",  "query"].edge_index.shape[1]
    print(
        f"  atoms={g.n_atoms:,}  query={g.n_query:,}  "
        f"cov={cov_e:,}  supp={supp_e:,}  aq={aq_e:,}  qq={qq_e:,}  "
        f"[{elapsed:.2f}s]"
    )
    graphs.append(g)

print(f"\nAll graphs built in {time.perf_counter() - t_total:.1f}s")

In [ ]:
# Fit ESP normalization across all three proteins and apply in-place
all_y    = torch.cat([g["query"].y for g in graphs])
esp_mean = float(all_y.mean())
esp_std  = float(all_y.std())
norm     = NormalizeESP(esp_mean, esp_std)

print(f"ESP stats  mean={esp_mean:.4f}  std={esp_std:.4f}")

for g in graphs:
    norm(g)        # normalizes g["query"].y in-place
    g.to(DEVICE)   # move all tensors to compute device

print(f"Graphs normalized and on {DEVICE}")

## Helper utilities

In [ ]:
# ── Per-protein plot colours ───────────────────────────────────────────────────
_PROTEIN_COLORS = {
    "AF-P01082-F1": "#e15759",   # red   (small)
    "AF-Q16613-F1": "#4e79a7",   # blue  (medium)
    "AF-B1WC58-F1": "#59a14f",   # green (large)
}
_PROTEIN_LABELS = {
    pid: pid.replace("AF-", "").replace("-F1", "") for pid in PROTEIN_IDS
}


def print_param_table(model: torch.nn.Module, model_name: str) -> int:
    """Print per-leaf-module parameter counts; return total."""
    rows = []
    for name, module in model.named_modules():
        if name == "":
            continue
        n = sum(p.numel() for p in module.parameters(recurse=False))
        if n > 0:
            rows.append((name, n))
    total = sum(p.numel() for p in model.parameters())

    col_w = max(len(r[0]) for r in rows) + 2
    sep   = "─" * (col_w + 16)
    print(f"\n{model_name} — parameters per module")
    print(sep)
    print(f"  {'Module':<{col_w}}  {'Parameters':>12}")
    print(sep)
    for name, n in rows:
        print(f"  {name:<{col_w}}  {n:>12,}")
    print(sep)
    print(f"  {'TOTAL':<{col_w}}  {total:>12,}")
    return total


def train_model(
    model: torch.nn.Module,
    graphs: list,
    model_name: str,
    n_epochs:       int   = N_EPOCHS,
    lr:             float = LR,
    pearson_weight: float = PEARSON_WEIGHT,
    clip_grad:      float = CLIP_GRAD,
) -> tuple:
    """
    Train model for n_epochs, one graph at a time (each protein = one batch).

    Peak VRAM is measured independently per protein: CUDA peak stats are reset
    immediately before each protein's forward pass and recorded after the
    optimizer step.  The max across all epochs is kept per protein.

    Returns:
        epoch_losses          — list[float], mean loss per epoch
        per_protein_losses    — dict[pid → list[float]], one value per epoch
        timing                — dict with timing, per_protein_peak_mb, etc.
    """
    loss_fn   = ESPLoss(pearson_weight=pearson_weight)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY
    )

    epoch_losses          = []
    per_protein_losses    = defaultdict(list)
    per_protein_peak_mb   = {g.protein_id: 0.0 for g in graphs}
    epoch_times           = []

    if _HAS_PSUTIL:
        ram_before = psutil.Process().memory_info().rss / 1e6

    hdr = f"  {'Epoch':>6}  {'Avg Loss':>10}  {'Min Loss':>10}  {'Max Loss':>10}  {'Time':>7}"
    print(hdr)
    print("  " + "─" * (len(hdr) - 2))

    for epoch in range(1, n_epochs + 1):
        t0 = time.perf_counter()
        model.train()
        losses_this_epoch = []

        for data in graphs:
            n_q   = data["query"].pos.shape[0]
            batch = torch.zeros(n_q, dtype=torch.long, device=DEVICE)

            # Reset peak tracker immediately before this protein's pass
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats(DEVICE)

            optimizer.zero_grad()
            pred   = model(data)
            target = data["query"].y
            loss   = loss_fn(pred, target, batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()

            # Record peak for this protein — keep the max across epochs
            if torch.cuda.is_available():
                peak = torch.cuda.max_memory_allocated(DEVICE) / 1e6
                per_protein_peak_mb[data.protein_id] = max(
                    per_protein_peak_mb[data.protein_id], peak
                )

            l = loss.item()
            losses_this_epoch.append(l)
            per_protein_losses[data.protein_id].append(l)

        avg = float(np.mean(losses_this_epoch))
        epoch_losses.append(avg)
        epoch_times.append(time.perf_counter() - t0)

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"  {epoch:>6}  {avg:>10.4f}  "
                f"{min(losses_this_epoch):>10.4f}  "
                f"{max(losses_this_epoch):>10.4f}  "
                f"{epoch_times[-1]:>6.2f}s"
            )

    timing = {
        "n_epochs":            n_epochs,
        "total_s":             sum(epoch_times),
        "mean_epoch_s":        float(np.mean(epoch_times)),
        "per_protein_peak_mb": per_protein_peak_mb,
    }
    if _HAS_PSUTIL:
        timing["ram_delta_mb"] = psutil.Process().memory_info().rss / 1e6 - ram_before

    return epoch_losses, dict(per_protein_losses), timing


def print_timing(timing: dict, model_name: str) -> None:
    """Print training timing summary."""
    print(f"\n── {model_name} — Training summary {'─'*28}")
    print(f"  Epochs        : {timing['n_epochs']}")
    print(f"  Total time    : {timing['total_s']:.1f} s")
    print(f"  Mean / epoch  : {timing['mean_epoch_s']:.2f} s")
    if "ram_delta_mb" in timing:
        sign = "+" if timing["ram_delta_mb"] >= 0 else ""
        print(f"  RAM delta     : {sign}{timing['ram_delta_mb']:.1f} MB")


def print_vram_table(timing: dict, model_name: str, n_params: int) -> None:
    """
    Print per-protein peak VRAM and estimated max batch size for common GPU sizes.

    Peak VRAM (measured) = model weights + gradients + optimizer states + graph
    tensors + intermediate activations for one protein.

    Model overhead is estimated as 4 × param bytes (float32):
      weights (1×) + gradients (1×) + AdamW m1 (1×) + AdamW m2 (1×)

    Per-protein variable cost = peak - model_overhead.
    Max batch size = floor((GPU_VRAM - model_overhead) / per_protein_variable)
    """
    if not torch.cuda.is_available():
        print("VRAM tracking requires CUDA — skipping.")
        return

    peaks = timing["per_protein_peak_mb"]   # pid → peak MB (max across epochs)

    # Model overhead: params + grads + AdamW m1 + m2  (all float32)
    model_overhead_mb = (n_params * 4 * 4) / 1e6

    print(f"\n── {model_name} — VRAM per protein {'─'*32}")
    print(f"  Model overhead  (weights + grads + AdamW m1/m2): {model_overhead_mb:.1f} MB")
    print()

    col_w = max(len(p) for p in peaks) + 2
    print(f"  {'Protein':<{col_w}}  {'Peak VRAM':>10}  {'Variable cost':>14}")
    print(f"  {'':─<{col_w}}  {'(MB)':>10}  {'(MB)':>14}")

    var_costs = {}
    for pid, peak_mb in peaks.items():
        var_mb = max(0.0, peak_mb - model_overhead_mb)
        var_costs[pid] = var_mb
        print(f"  {pid:<{col_w}}  {peak_mb:>10.1f}  {var_mb:>14.1f}")

    # Batch capacity table
    print()
    print(f"  Estimated max batch size per GPU  (floor((GPU - overhead) / variable))")
    pid_list = list(peaks.keys())
    labels   = [_PROTEIN_LABELS[p] for p in pid_list]
    print(f"  {'GPU':>8}  " + "  ".join(f"{lb:>12}" for lb in labels))
    print("  " + "─" * (10 + 14 * len(pid_list)))
    for gb in GPU_SIZES_GB:
        avail_mb = gb * 1024 - model_overhead_mb
        row = f"  {gb:>5} GB  "
        for pid in pid_list:
            v = var_costs[pid]
            n = int(avail_mb // v) if v > 0 and avail_mb > 0 else 0
            row += f"  {n:>12}"
        print(row)


def plot_loss(
    epoch_losses: list,
    per_protein_losses: dict,
    model_name: str,
) -> None:
    """Two-panel loss plot: average + per-protein breakdown."""
    epochs = range(1, len(epoch_losses) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    fig.suptitle(model_name, fontsize=13, fontweight="bold")

    # Left — average loss
    ax = axes[0]
    ax.plot(epochs, epoch_losses, linewidth=2, color="steelblue",
            marker="o", markersize=3)
    ax.set_title("Average loss (all proteins)")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

    # Right — per-protein
    ax = axes[1]
    for pid, losses in per_protein_losses.items():
        ax.plot(
            range(1, len(losses) + 1), losses,
            label=_PROTEIN_LABELS[pid],
            color=_PROTEIN_COLORS.get(pid, "gray"),
            linewidth=1.8,
        )
    ax.set_title("Loss per protein")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend(title="Protein", fontsize=9)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

    plt.tight_layout()
    plt.show()


print("Helpers defined.")

---
## 2. DistanceESPN

Mean-aggregation message passing on all four edge types (cov, supp, AQ, QQ).  
Geometry is encoded purely via Gaussian RBF features — no attention, no coordinate updates.

In [ ]:
dist_model = DistanceESPN(
    hidden_dim        = HIDDEN_DIM,
    n_rbf             = N_RBF,
    n_cov_supp_rounds = N_COV_SUPP_ROUNDS,
    n_aq_rounds       = N_AQ_ROUNDS,
    n_qq_rounds       = N_QQ_ROUNDS,
).to(DEVICE)

n_params_dist = print_param_table(dist_model, "DistanceESPN")

### Training

In [ ]:
dist_epoch_losses, dist_protein_losses, dist_timing = train_model(
    dist_model, graphs, model_name="DistanceESPN"
)

In [ ]:
print_timing(dist_timing, "DistanceESPN")

In [ ]:
print_vram_table(dist_timing, "DistanceESPN", n_params_dist)

In [ ]:
plot_loss(dist_epoch_losses, dist_protein_losses, "DistanceESPN")

---
## 3. AttentionESPN

Identical to DistanceESPN for Stages 1 (atom encoder, cov + supp) and 3 (QQ refinement).  
Stage 2 replaces mean aggregation with **multi-head cross-attention** on AQ edges,
with a per-head RBF geometry bias added to the attention scores.

In [ ]:
attn_model = AttentionESPN(
    hidden_dim        = HIDDEN_DIM,
    n_rbf             = N_RBF,
    n_heads           = N_HEADS,
    n_cov_supp_rounds = N_COV_SUPP_ROUNDS,
    n_aq_rounds       = N_AQ_ROUNDS,
    n_qq_rounds       = N_QQ_ROUNDS,
).to(DEVICE)

n_params_attn = print_param_table(attn_model, "AttentionESPN")

### Training

In [ ]:
attn_epoch_losses, attn_protein_losses, attn_timing = train_model(
    attn_model, graphs, model_name="AttentionESPN"
)

In [ ]:
print_timing(attn_timing, "AttentionESPN")

In [ ]:
print_vram_table(attn_timing, "AttentionESPN", n_params_attn)

In [ ]:
plot_loss(attn_epoch_losses, attn_protein_losses, "AttentionESPN")

---
## 4. Comparison

Side-by-side summary and combined loss curves.

In [ ]:
W = 25
print(f"\n{'Metric':<{W}} {'DistanceESPN':>14} {'AttentionESPN':>14}")
print("─" * (W + 30))
print(f"{'Parameters':<{W}} {n_params_dist:>14,} {n_params_attn:>14,}")
print(f"{'Training time (s)':<{W}} {dist_timing['total_s']:>14.1f} {attn_timing['total_s']:>14.1f}")
print(f"{'Mean / epoch (s)':<{W}} {dist_timing['mean_epoch_s']:>14.2f} {attn_timing['mean_epoch_s']:>14.2f}")
if "ram_delta_mb" in dist_timing:
    print(f"{'RAM delta (MB)':<{W}} {dist_timing['ram_delta_mb']:>+14.1f} {attn_timing['ram_delta_mb']:>+14.1f}")
print(f"{'Final avg loss':<{W}} {dist_epoch_losses[-1]:>14.4f} {attn_epoch_losses[-1]:>14.4f}")
print(f"{'Best avg loss':<{W}} {min(dist_epoch_losses):>14.4f} {min(attn_epoch_losses):>14.4f}")
print(f"{'Loss drop (first→last)':<{W}} {dist_epoch_losses[0]-dist_epoch_losses[-1]:>14.4f} {attn_epoch_losses[0]-attn_epoch_losses[-1]:>14.4f}")

if torch.cuda.is_available():
    print()
    print(f"{'Peak VRAM per protein':<{W}} {'DistanceESPN':>14} {'AttentionESPN':>14}")
    print("─" * (W + 30))
    for pid in PROTEIN_IDS:
        d = dist_timing["per_protein_peak_mb"][pid]
        a = attn_timing["per_protein_peak_mb"][pid]
        label = _PROTEIN_LABELS[pid]
        print(f"  {label:<{W-2}} {d:>13.1f}  {a:>13.1f}  MB")

In [ ]:
epochs = range(1, N_EPOCHS + 1)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    epochs, dist_epoch_losses,
    label="DistanceESPN", linewidth=2,
    color="steelblue", marker="o", markersize=3,
)
ax.plot(
    epochs, attn_epoch_losses,
    label="AttentionESPN", linewidth=2,
    color="darkorange", marker="s", markersize=3, linestyle="--",
)
ax.set_title("Training loss — DistanceESPN vs AttentionESPN", fontsize=12)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss  (MSE + Pearson)")
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()